### 알고리즘 A/B 테스트 데이터 분석


 - 알고리즘 A/B 테스트 데이터셋 : datasets/abtest/algorithm_ab_test.csv
  
 - 분석 준비
   - 데이터셋 구조 파악 (컬럼 설명)
     * user_id: 사용자 ID
  
     * timestamp: 테스트 실행 시간
     * group: 테스트 그룹 (A/B)
     * version: 알고리즘 버전 (v1/v2)
     * conversion: 전환 여부 (0/1)
     * revenue: 수익 금액
   - pandas로 데이터셋 로드 및 전처리
  
   - A/B 테스트 통계적 분석 수행

### 0. Preprocessing

In [55]:
import plotly.express as px
import pandas as pd

In [16]:
data = pd.read_csv("/Users/jisupark_1/workspace/star_track_python/datasets/abtest/algorithm_ab_test.csv")

data.head()

,uid,ts,grp,algorithm,converted,amount
0,851227,2025-01-21 22:11:48.556739,A,v1,0,0
1,804351,2025-01-12 08:01:45.159739,A,v1,0,0
2,661713,2025-01-11 16:55:06.154213,B,v2,0,11762
3,853664,2025-01-08 18:28:03.143765,B,v2,0,14987
4,865098,2025-01-21 01:52:26.210827,A,v1,1,13058


#### 0-1. 결측치 처리
- null 값 없음 -> 결측치 처리 불필요

In [17]:
data.isnull().sum()

uid          0
ts           0
grp          0
algorithm    0
converted    0
amount       0
dtype: int64

#### 0-2. ts (timestamp) column dtype 변환
- pd.to_datetime('col')
- Object 형식 -> datatime64 형식

In [45]:
## Before -> dtype('O') -> Object 형식
data['ts'].dtype


dtype('<M8[ns]')

In [46]:
## After -> dtype('<M8[ns]') ->  datetime64[ns] 타입

data['ts'] = pd.to_datetime(data['ts'])

data['ts'].dtype

dtype('<M8[ns]')

#### 0-3. ts 기준 sorting
- data.sort_values('col')

In [47]:
data = data.sort_values('ts')

data.head()

,uid,ts,grp,algorithm,converted,amount,log_amount
325747,425323,2025-01-02 13:32:15.234051,C,v3,0,12780,9.455715
296988,110858,2025-01-02 13:33:03.767329,C,v3,0,12333,9.420115
420164,179783,2025-01-02 13:36:06.600508,C,v3,0,10712,9.279213
332968,437507,2025-01-02 13:37:28.246588,C,v3,1,11101,9.314881
384759,192894,2025-01-02 13:37:52.169597,C,v3,1,11191,9.322955


#### 0-4. 이상치 처리 -> 로그 변환
- 미결제 고객이 많고 소수의 결제값이 매우 큰 고객이 존재하는 것으로 확인됨 -> 분포가 왜곡되어있음
- 이상치 제거보다 분포 왜곡을 완화하기 위해 로그 변환 수행 -> col = log_amount

In [48]:
import numpy as np

data['log_amount'] = np.log1p(data['amount'])  # log(1 + x)


#### 0-5. 중복 사용자 확인 및 제거 
- AB Test 를 진행할 것이므로 / 현재 데이터셋에는 이미 그룹이 나뉘어져 있으나 혹시 모르니 진행하였음

In [49]:
data_unique = data.drop_duplicates(subset='uid')

data_unique.head()


,uid,ts,grp,algorithm,converted,amount,log_amount
325747,425323,2025-01-02 13:32:15.234051,C,v3,0,12780,9.455715
296988,110858,2025-01-02 13:33:03.767329,C,v3,0,12333,9.420115
420164,179783,2025-01-02 13:36:06.600508,C,v3,0,10712,9.279213
332968,437507,2025-01-02 13:37:28.246588,C,v3,1,11101,9.314881
384759,192894,2025-01-02 13:37:52.169597,C,v3,1,11191,9.322955


### 1. Metric 

#### 1-1. 그룹별 전환율 계산
- A 그룹의 전환율: 0.120274
- B 그룹의 전환율: 0.118855
- C 그룹의 전환율: 0.119947

=> 각 그룹의 전환율이 큰 차이를 보이지 않음

In [50]:
group_convers = data_unique.groupby('grp')['converted'].mean().reset_index()
group_convers.columns = ['grp', 'conversion_rate']

print(group_convers)

  grp  conversion_rate
0   A         0.120422
1   B         0.118845
2   C         0.119947


#### 1-2. 그룹별 결제율 계산
- A 그룹의 결제율: 0.120274
- B 그룹의 결제율: 1
- C 그룹의 결제율: 1

- => A 그룹은 전환율과 결제율이 같음. 즉, 전환된 유저는 결제를 함
- => B, C 그룹은 전환율과 결제율이 다름. 즉, 그룹 내 모든 유저가 결제를 했으나 전환은 별도의 기준으로 판단을 함
- => 전환 되었다의 기준이 결제 했다는 아님

In [51]:
group_rev_r = data_unique.groupby('grp').apply(
    lambda df: (df['amount'] > 0).mean()
).reset_index(name='pay_rate')

print(group_rev_r)

  grp  pay_rate
0   A  0.120422
1   B  1.000000
2   C  1.000000


/var/folders/pt/8357krnj4tv48kdjx9mrhd3r0000gp/T/ipykernel_48513/3469655010.py:1: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



#### 1-3. 실제 전환자 대상 그룹별 결제금액
- 전환자 기준으로는 세 그룹이 비슷한 기술통계량을 보유함

In [52]:
# 전환자만 필터링
converted_data = data[data['converted'] == 1]

# 그룹별 기술통계 요약
summary_converted = converted_data.groupby('grp')['amount'].describe()
print(summary_converted)


       count          mean          std     min      25%      50%      75%  \
grp                                                                          
A    17723.0  12537.662980  1099.987381  8437.0  11791.0  12545.0  13277.5   
B    17514.0  12745.035400  1160.648727  8463.0  11960.0  12745.0  13522.0   
C    17628.0  12751.578341  1153.871314  8511.0  11966.0  12739.0  13536.0   

         max  
grp           
A    17202.0  
B    16830.0  
C    16928.0  


#### 1-4. 그룹별 알고리즘 버전에 따른 사용자 수
- A 그룹은 알고리즘 버젼 1, 2를 사용하나 v1 유저가 더 많음
- B 그룹은 알고리즘 버젼 1, 2를 사용하나 v2 유저가 더 많음
- C 그룹은 알고리즘 버젼 3을 사용하는 유저

In [32]:
# 그룹별 알고리즘 버전별 사용자 수 카운트
user_counts = data_unique.groupby(['grp', 'algorithm'])['uid'].nunique().reset_index(name='user_count')
print(user_counts)


  grp algorithm  user_count
0   A        v1      144197
1   A        v2        1005
2   B        v1        1038
3   B        v2      144290
4   C        v3      146956


### 2. 통계적 수치 확인 



#### 2-1. describe()
    - uid column은 유저 아이디이므로 통계적 수치를 보는 것이 의미없음
    - converted 는 전환여부를 나타내는 binary 변수를 담은 column
    - mean 이 약 0.12 즉, 전환율 12%
    - amount 는 각 유저가 해당 서비스에 결제한 금액을 나타내는 column
        - mean(8,998) < median(12,103) -> 구매 x 고객이 다수 존재하여 그런 듯함
            - right-skewed -> 소수의 고액 결제 고객
            - 유저 분리가 필요할 수 있을 듯 함
    - log_amount 는 amount 의 왜곡을 완화하기 위해 log 변환한 column
        -  mean(6.68), median(9.40) -> 로그 변환 후에도 여전히 편향 존재하지만 완화됨

In [9]:
data_unique.describe()

,uid,ts,converted,amount,log_amount
count,437486.000000,437486,437486.000000,437486.000000,437486.000000
mean,646002.725431,2025-01-13 12:57:27.599814144,0.119739,9014.191096,6.688413
min,100002.000000,2025-01-02 13:32:15.234051,0.000000,0.000000,0.000000
25%,494913.500000,2025-01-08 01:18:11.390169088,0.000000,0.000000,0.000000
50%,708207.500000,2025-01-13 12:17:16.071379456,0.000000,12108.000000,9.401704
75%,827185.750000,2025-01-19 01:01:06.690692352,0.000000,13163.000000,9.485241
max,946122.000000,2025-01-24 13:50:19.152664,1.000000,17778.000000,9.785773
std,230921.628912,NaN,0.324656,5870.397171,4.296820


#### 2-2. group 별 describe()
    - group 별 사용자 분포가 고르게 되어있음 
        - A: 145,273 
        - B: 145,275
        - C: 146,956
    - group A 에 구매력이 낮은 유저가 주로 분포하고 있음을 알 수 있음
    - group B 와 C 는 비슷한 기술통계량을 가짐


In [10]:
data_unique[data_unique['grp'] == 'A'].describe()

,uid,ts,converted,amount,log_amount
count,145273.000000,145273,145273.000000,145273.000000,145273.000000
mean,788249.769820,2025-01-13 12:48:10.726887680,0.120422,1509.740571,1.135893
min,630125.000000,2025-01-02 13:42:15.234051,0.000000,0.000000,0.000000
25%,709371.000000,2025-01-08 01:12:10.343552,0.000000,0.000000,0.000000
50%,788215.000000,2025-01-13 12:03:10.013416960,0.000000,0.000000,0.000000
75%,867255.000000,2025-01-19 00:52:53.843490048,0.000000,0.000000,0.000000
max,946121.000000,2025-01-24 13:41:54.460509,1.000000,17202.000000,9.752839
std,91285.484196,NaN,0.325455,4098.062118,3.070054


In [11]:
data_unique[data_unique['grp'] == 'B'].describe()

,uid,ts,converted,amount,log_amount
count,145257.000000,145257,145257.000000,145257.000000,145257.000000
mean,788001.028529,2025-01-13 12:08:46.869525760,0.118845,12744.710375,9.448812
min,630123.000000,2025-01-02 13:42:05.378582,0.000000,7739.000000,8.954157
25%,708878.000000,2025-01-08 00:15:28.125916928,0.000000,11969.000000,9.390159
50%,788041.000000,2025-01-13 10:46:16.180090112,0.000000,12745.000000,9.452973
75%,866902.000000,2025-01-19 00:11:31.544743936,0.000000,13521.000000,9.512073
max,946122.000000,2025-01-24 13:41:44.097174,1.000000,17777.000000,9.785717
std,91165.205128,NaN,0.323607,1152.371873,0.091355


In [12]:
data_unique[data_unique['grp'] == 'C'].describe()

,uid,ts,converted,amount,log_amount
count,146956.000000,146956,146956.000000,146956.000000,146956.000000
mean,365028.131727,2025-01-13 13:54:45.058102784,0.119947,12745.307908,9.448858
min,100002.000000,2025-01-02 13:32:15.234051,0.000000,7739.000000,8.954157
25%,232346.750000,2025-01-08 02:30:14.465915392,0.000000,11969.000000,9.390159
50%,364337.000000,2025-01-13 13:51:17.617946624,0.000000,12746.000000,9.453051
75%,497892.500000,2025-01-19 01:59:27.411490304,0.000000,13522.000000,9.512147
max,630120.000000,2025-01-24 13:50:19.152664,1.000000,17778.000000,9.785773
std,153019.201711,NaN,0.324901,1152.469016,0.091361


#### Welch's t-test 통계 검정
- 선택 이유: A, B 그룹간의 등분산성을 가정하기 어려웠음. 이에 등분산성을 가정하지 않는 Welch's t-test 를 이용

In [53]:
from scipy.stats import ttest_ind

# 그룹별 전환자 추출
group_A = data[(data['grp'] == 'A') & (data['converted'] == 1)]['amount']
group_B = data[(data['grp'] == 'B') & (data['converted'] == 1)]['amount']
group_C = data[(data['grp'] == 'C') & (data['converted'] == 1)]['amount']

# Welch's t-test (equal_var=False)
t_ab, p_ab = ttest_ind(group_A, group_B, equal_var=False)
t_ac, p_ac = ttest_ind(group_A, group_C, equal_var=False)
t_bc, p_bc = ttest_ind(group_B, group_C, equal_var=False)

print(f"A vs B: t={t_ab:.4f}, p={p_ab:.4f}")
print(f"A vs C: t={t_ac:.4f}, p={p_ac:.4f}")
print(f"B vs C: t={t_bc:.4f}, p={p_bc:.4f}")


A vs B: t=-17.2102, p=0.0000
A vs C: t=-17.8387, p=0.0000
B vs C: t=-0.5299, p=0.5962


### 3. 시각화

##### 3-1. 그룹별 결제 금액 분포
- 결제한 사용자 기준으로 보면 결제 단가 자체는 알고리즘에 따라 거의 차이가 없음
- 사용자 간 결제 성향 분포가 세 그룹 간 거의 유사

In [70]:

fig = px.box(data_unique[data_unique['amount'] > 0], 
             x='grp', 
             y='amount',
             color='grp',
             title='📦 그룹별 결제 금액 분포 (Boxplot)',
             labels={'grp': '그룹', 'amount': '결제 금액'})
fig.update_layout(yaxis_tickformat=",")
fig.show()


#### 3-2. 그룹별 수익 비교
- A 그룹은 B/C 대비 수익 규모가 약 1/8 ~ 1/9 수준

In [ ]:

# 그룹별 수익 합계
group_revenue = data_unique.groupby('grp')['amount'].sum().reset_index(name='total_revenue')

# 막대그래프 시각화
fig = px.bar(group_revenue, 
             x='grp', 
             y='total_revenue',
             color='grp',
             title='💰 그룹별 총 수익 비교',
             labels={'grp': '그룹', 'total_revenue': '총 수익'},
             text_auto=True)
fig.update_layout(yaxis_tickformat=",")
fig.show()


#### 3-3. 시간에 따른 전환율 추이

- A, B, C 모든 그룹의 전환율은 대체로 11.5% ~ 12.5% 사이에 존재
- 특정 알고리즘이 전환율에 주는 영향이 통계적으로 유의하지 않을 가능성 높음

In [ ]:

# ts 컬럼을 datetime 형식으로 변환
data['ts'] = pd.to_datetime(data['ts'])

# 그룹별 + 날짜별 전환율 계산
group_daily_conv = (
    data.groupby([data['ts'].dt.date, 'grp'])['converted']
    .agg(['sum', 'count'])
    .reset_index()
    .rename(columns={'sum': 'converted', 'count': 'total', 'ts': 'date'})
)

group_daily_conv['conversion_rate'] = group_daily_conv['converted'] / group_daily_conv['total']

# 시각화
fig = px.line(
        group_daily_conv, 
        x='date', 
        y='conversion_rate', 
        color='grp',
        title='📊 그룹별 일별 전환율 변화',
        labels={'date': '날짜', 'conversion_rate': '전환율', 'grp': '그룹'})

fig.update_traces(mode='lines+markers')
fig.update_layout(yaxis_tickformat=".1%", xaxis_title="날짜", yaxis_title="전환율")
fig.show()


#### 3-4. 시간에 따른 그룹별 revenue 추이
- A 그룹은 수익이 1천만 미만으로 아주 낮은 수준
- B, C 그룹은 약 8천만 ~ 9천만 단위로 매우 높은 수준

In [67]:
import pandas as pd
import plotly.express as px

# 날짜 컬럼 생성
data['ts'] = pd.to_datetime(data['ts'])
data['date'] = data['ts'].dt.date

# 일별 그룹별 revenue 집계
daily_revenue = (
    data.groupby(['date', 'grp'])['amount']
    .sum()
    .reset_index(name='daily_revenue')
)

# 시각화
fig = px.line(daily_revenue, x='date', y='daily_revenue', color='grp',
              title='💵 그룹별 일별 Revenue 추이',
              labels={'date': '날짜', 'daily_revenue': '총 결제 금액', 'grp': '그룹'})
fig.update_traces(mode='lines+markers')
fig.update_layout(yaxis_tickformat=",", xaxis_title="날짜", yaxis_title="Revenue")
fig.show()


### 4. 가설 설정 및 검증

#### 4-1. 알고리즘 버젼에 따른 전환율 차이
- 귀무가설(H₀): 알고리즘 v1을 사용하는 유저와 v2를 사용하는 유저의 전환율(결제율)은 차이가 없다.

- 대립가설(H₁): 알고리즘 v1 유저의 전환율이 v2 유저보다 낮다.

##### 결과 해석
- Z-statistic: 1.3683
- P-value: 0.9144
- p-val 이 유의수준 0.05보다 훨씬 크므로 귀무가설을 기각할 수 없음. 즉, v1 유저가 전환율이 v2 보다 낮다고 볼 충분한 통계적 증거가 없음

In [34]:
# A: v1 사용자 (알고리즘 1)
group_v1 = data[(data['algorithm'] == 'v1') & (data['grp'].isin(['A', 'B']))]

# B: v2 사용자 (알고리즘 2)
group_v2 = data[(data['algorithm'] == 'v2') & (data['grp'].isin(['A', 'B']))]


In [35]:
from statsmodels.stats.proportion import proportions_ztest

# 성공 수, 관측 수
success = [group_v1['converted'].sum(), group_v2['converted'].sum()]
nobs = [len(group_v1), len(group_v2)]

# 단측 z-test: H1: prop_v1 < prop_v2
z_stat, p_value = proportions_ztest(success, nobs, alternative='smaller')

print(f"Z-statistic: {z_stat:.4f}")
print(f"P-value: {p_value:.4f}")


Z-statistic: 1.3683
P-value: 0.9144


#### 4-2. 그룹별 전환된 사람의 결제 금액 차이 (Welch’s t-test)
- 결제금액 T-통계량: -23.4239
- p-value: 0.0000
- A 그룹의 평균 결제 금액이 B 그룹보다 유의미하게 낮다

In [37]:
from scipy.stats import ttest_ind

# 결제한 유저만 추출
amount_A = group_A[group_A['amount'] > 0]['amount']
amount_B = group_B[group_B['amount'] > 0]['amount']

t_stat, p_val_t = ttest_ind(amount_A, amount_B, equal_var=False)
print(f"결제금액 T-통계량: {t_stat:.4f}, p-value: {p_val_t:.4f}")


결제금액 T-통계량: -23.4239, p-value: 0.0000


#### 해석
1. 전환율 Z-test
- A와 B 그룹 간 전환율 차이는 통계적으로 유의하지 않음
- 전환 자체를 유도하는 힘은 알고리즘에 큰 차이 없음
2. ARPU (유저당 평균 수익)
- A 그룹 대비 B 그룹의 전체 수익성(ARPU)이 현저히 높음
- 환율은 비슷하지만, 전환된 유저가 결제를 많이 하고 금액도 큼
3. 고액 결제자 비율 (>15,000원)
- B 그룹에는 A 그룹보다 고액 결제자 비율이 압도적으로 높음
- v2 알고리즘은 고가 상품 구매에 더 유리
4. 요일별 전환율 (Chi2)
- 요일에 따른 그룹 간 전환율 차이는 없음
5. 로그 결제 금액 비교
- 결제 금액 분포 자체가 B 그룹이 더 높게 쏠려 있음
- 소수 고액 유저 영향이 아니라 구조적 차이

In [ ]:
from scipy.stats import ttest_ind, chi2_contingency
from statsmodels.stats.proportion import proportions_ztest

In [ ]:
# 날짜 및 요일 처리
data_unique['ts'] = pd.to_datetime(data_unique['ts'])
data_unique['weekday'] = data_unique['ts'].dt.day_name()

# 그룹 필터
group_A = data_unique[(data_unique['grp'] == 'A') & (data_unique['algorithm'] == 'v1')]
group_B = data_unique[(data_unique['grp'] == 'B') & (data_unique['algorithm'] == 'v2')]

# 1. 전환율 비교
z1, p1 = proportions_ztest(
    [group_A['converted'].sum(), group_B['converted'].sum()],
    [len(group_A), len(group_B)]
)

# 2. ARPU 비교 (전체 유저 기준)
t3, p3 = ttest_ind(
    group_A['amount'],
    group_B['amount'],
    equal_var=False
)

# 3. 고액 결제자 비율 (15,000 초과)
z4, p4 = proportions_ztest(
    [(group_A['amount'] > 15000).sum(), (group_B['amount'] > 15000).sum()],
    [len(group_A), len(group_B)]
)

# 4. 요일별 전환율 차이 (카이제곱 검정)
cross = pd.crosstab(data_unique['grp'], data_unique['weekday'])
chi2, p5, _, _ = chi2_contingency(cross)

# 5. 로그 결제금액 비교
t6, p6 = ttest_ind(
    group_A[group_A['amount'] > 0]['log_amount'],
    group_B[group_B['amount'] > 0]['log_amount'],
    equal_var=False
)

# 결과 요약 출력
results = {
    "1. 전환율 Z-test": (z1, p1),
    "2. ARPU t-test": (t3, p3),
    "3. 고액 결제자 비율 Z-test": (z4, p4),
    "4. 요일별 전환율 Chi2": (chi2, p5),
    "5. 로그 결제금액 t-test": (t6, p6)
}

for test_name, (stat, pval) in results.items():
    print(f"{test_name} → 통계량: {stat:.4f}, p-value: {pval:.4f}")


/var/folders/pt/8357krnj4tv48kdjx9mrhd3r0000gp/T/ipykernel_48513/1607183582.py:2: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/pt/8357krnj4tv48kdjx9mrhd3r0000gp/T/ipykernel_48513/1607183582.py:3: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



1. 전환율 Z-test → 통계량: 1.3954, p-value: 0.1629
2. ARPU t-test → 통계량: -1002.6321, p-value: 0.0000
3. 고액 결제자 비율 Z-test → 통계량: -55.8809, p-value: 0.0000
4. 요일별 전환율 Chi2 → 통계량: 5.0706, p-value: 0.9556
5. 로그 결제금액 t-test → 통계량: -22.6638, p-value: 0.0000


### 5. 결론

- 실험군(B그룹)의 알고리즘(v2)이 전환율은 기존(A그룹, v1)과 유의미한 차이가 없지만, 매우 강력한 수익성 향상을 보였음
- 추가로 비열등성 검정 결과, B의 전환율이 A 보다 약간 낮긴 하지만 크게 유의미하지 않음. 즉, 비열등. 
- 따라서 만약 새로운 알고리즘(B)이 운영 비용이 더 저렴하거나, 유지 보수가 쉬운 구조라면, 비즈니스적으로 채택할 명분 충분

### 6. 추가 검정 - 비열등성 검정
- 전환율이 유지되었으며, 동시에 수익성이 좋아졌다는 실험 성과


- 실험군(B)의 전환율이 기존(A)보다 약간 낮긴 하지만, 우리가 설정한 "허용 가능한 최대 차이(−1%p)"보다 차이가 작음
- 실험군은 기존보다 유의미하게 나쁘지 않다, 즉 비열등하다고 볼 수 있음

In [74]:
import numpy as np
from statsmodels.stats.proportion import proportion_effectsize
from statsmodels.stats.power import NormalIndPower

In [75]:

# 그룹별 전환율
p1 = 0.1204  # A 그룹 전환율 
p2 = 0.1188  # B 그룹 전환율 

# 비열등성 마진 설정 (예: -0.01)
non_inferiority_margin = 0.01

# 효과 크기 (비열등성 기준 하의 Cohen's h)
effect_size_ni = proportion_effectsize(p1, p1 - non_inferiority_margin)

# 검정력 분석기 설정
alpha = 0.05
power = 0.8
analysis = NormalIndPower()

# 필요한 샘플 크기 계산
sample_size = analysis.solve_power(effect_size=effect_size_ni, power=power, alpha=alpha, ratio=1)

# 실제 차이 확인 및 비열등 판단
actual_diff = p2 - p1
is_non_inferior = actual_diff > -non_inferiority_margin

print(f"기존 전환율: {p1:.4f}")
print(f"실험 전환율: {p2:.4f}")
print(f"비열등성 마진: {-non_inferiority_margin}")
print(f"전환율 차이: {actual_diff:.4f}")
print("비열등성 판단 결과:", "✅ 비열등함" if is_non_inferior else "❌ 비열등하지 않음")
print(f"비열등성 검정에 필요한 최소 표본 수 (그룹당): {np.ceil(sample_size):.0f}")


기존 전환율: 0.1204
실험 전환율: 0.1188
비열등성 마진: -0.01
전환율 차이: -0.0016
비열등성 판단 결과: ✅ 비열등함
비열등성 검정에 필요한 최소 표본 수 (그룹당): 16018
